# DSA Drill - Longest Substring Without Repeating Characters (LC 3 - Medium)

> Given a string `s`, return the length of the **longest substring** (contiguous) that contains **no repeating characters**.

**Examples**
1. `s = "abcabcbb"` -> `3`  ("abc")
2. `s = "bbbbb"` -> `1`  ("b")
3. `s = "pwwkew"` -> `3`  ("wke") - classic trap: "pwke" is a subsequence, NOT a substring
4. `s = "abba"` -> `?`  - work it out by hand; this one is a landmine

**Constraints:** `0 <= len(s) <= 5 * 10^4`; `s` may contain letters, digits, symbols and spaces.

**Protocol (20:00 cap)**
- 0-2 min: clarify out loud
- 2-6 min: verbalize brute force + complexity, then say WHY a sliding window is valid here (the family change)
- 6-18 min: code it
- 18-20 min: run the examples + the `"abba"` case + one edge case of your own

**Landmines to answer in the post-mortem**
1. **Why does a sliding window work here but NOT in LC 560/974?** (Hint: what happens to validity when you EXTEND a window in each problem?)
2. `"abba"` - what is the answer, and why must the last-seen-index version use `left = max(left, last_seen[c] + 1)` instead of `left = last_seen[c] + 1`?
3. Two valid implementations: (a) set + shrink-while-loop, (b) dict of last-seen indices. Which is easier to reason about, which touches each character once? Give the complexity of each.

One hint allowed (costs the no-hints badge).

## Intuition (plain): maintain ONE window, don't count pairs

Family A (LC 560/974) counted **pairs of moments**. LC 3 is a different game: keep **one window** - a `left` pointer and a `right` pointer - and slide it.

**The two moves**
1. **Expand** - move `right` forward, always. One new character joins the window.
2. **Repair** - if that new character makes the window invalid (it now repeats one already inside), move `left` forward until the window is valid again.

**Why this is legal here and not in LC 560/974 - the monotonicity test**
- LC 3: if a window already has a duplicate, **adding more characters can never fix it**. Broken stays broken -> the only repair is to shrink from the left. Validity is *monotone*.
- LC 560/974: the rule was "sum equals k" / "sum divisible by k". A new element can **both break and fix** it (negatives especially) - so shrinking can't be justified, and we counted pairs instead.

**What you track:** the best valid window seen so far, whose length is `right - left + 1` (watch the +1).

**Picture for `"abcabcbb"`**
```
a b c a b c b b
^     ^
left  right      window "abca" is broken (two a's) -> repair = move left past the first 'a'
```

In [ ]:
# ---- V2 (optimal): one pass + dict of last-seen indices -- O(n) time ----
def length_of_longest_substring(s: str) -> int:
    """Return the length of the longest substring without repeating characters."""
    last = {}          # character -> last index where it appeared
    left = 0           # start of the current window
    best = 0
    for right, c in enumerate(s):
        # max() is essential - see the "abba" landmine: last[c] can sit BEFORE left,
        # and left must never move backwards.
        left = max(left, last.get(c, -1) + 1)
        last[c] = right
        best = max(best, right - left + 1)
    return best


# ---- V1 (your version): expand right always, shrink left while invalid ----
# Correct (verified against brute force). Each character is added once and removed
# at most once -> O(n) time as well; easiest version to argue in an interview.
def length_of_longest_substring_v1(s: str) -> int:
    if len(s) == 0:
        return 0
    counts = {}
    i = 0
    maxx = 0
    for j in range(len(s)):
        c = s[j]
        counts[c] = counts.get(c, 0) + 1
        while counts[c] > 1:
            counts[s[i]] -= 1
            i += 1
        maxx = max(maxx, j - i + 1)
    return maxx


# ---- The "abba" trap, for the record: NO max() at the left update ----
# def no_max(s):
#     last = {}; left = best = 0
#     for right, c in enumerate(s):
#         if c in last:
#             left = last[c] + 1        # BUG: left can jump BACKWARDS (2 -> 1 on "abba")
#         last[c] = right
#         best = max(best, right - left + 1)
#     return best                       # returns 3 for "abba"; truth is 2

In [23]:
length_of_longest_substring("abcabcbb")

3

In [24]:
# Reference (trustworthy brute force) - do not edit
def _brute(s):
    best = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            if s[j] in seen:
                break
            seen.add(s[j])
            best = max(best, j - i + 1)
    return best

assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("abba") == _brute("abba")   # landmine 2
assert length_of_longest_substring("") == 0

import random
random.seed(5)
for _ in range(300):
    n = random.randint(0, 12)
    s = "".join(random.choice("abc") for _ in range(n))
    assert length_of_longest_substring(s) == _brute(s), s
print("examples + fuzz 300 (small alphabet) == brute force OK")

examples + fuzz 300 (small alphabet) == brute force OK


In [25]:
# Edge cases - run me and say WHY out loud for each
assert length_of_longest_substring("") == 0          # empty string
assert length_of_longest_substring("a") == 1         # single character
assert length_of_longest_substring("aaaa") == 1      # all the same
assert length_of_longest_substring("abcdef") == 6    # all distinct
assert length_of_longest_substring(" ") == 1         # whitespace is a character
assert length_of_longest_substring("dvdf") == 3      # classic: "vdf", not "dv"
assert length_of_longest_substring("a b") == 3       # a space is just another character
assert length_of_longest_substring("   ") == 1       # 3 spaces = the SAME character repeating
print("edge cases pass OK")

edge cases pass OK


## Post-mortem (model answers - rewrite in your own words)

- **Brute force:** for every start index, extend while characters stay unique -> O(n^2) time, O(alphabet) space. Reference lives in the `tests` cell.
- **Landmine 1 - monotonicity (why a window is legal here, and was not in LC 560/974):** the property is "no duplicates inside the window". If a window is broken, adding more characters can NEVER repair it - so shrinking from the left is the only repair, and expanding right greedily is safe. Broken-ness is monotone. In LC 560/974 the property was "sum = k" / "sum divisible by k", which a new element can both break and fix (negatives) - no grow/shrink argument exists there, which is exactly why those needed pair counting instead.
- **Landmine 2 - `"abba"`:** answer = **2** ("ab"). `last[c]` is the globally last seen index, which may sit BEFORE the current `left`. On `"abba"`: at right=3 ('a'), `last['a']=0` -> `left = 0+1 = 1`, jumping backwards past the window start (left was 2 after the repeated 'b'), which re-admits the 'b' and returns 3 (wrong). `left = max(left, last[c]+1)` keeps `left` monotone (the `if last[c] >= left` guard is an equivalent form).
- **Landmine 3 - two implementations:** (a) set/counts + shrink-while-loop - easiest to prove correct, each character touched at most twice -> O(n); (b) last-seen dict - single pass, one touch per character, `left` jumps straight past the duplicate -> O(n) time, O(min(n, alphabet)) space. Interview arc: present (a) first, then offer (b) as the optimization.
- **Complexity (V2):** O(n) time, O(min(n, alphabet)) space.
- **Measured:** all 14 example/edge inputs agree across brute / V1 / V2; fuzz **500/500** over letters + digits + symbols + SPACES + uppercase; the no-`max` variant returns 3 on `"abba"` (wrong); n=100,000: V2 **approx 11.5 ms**, V1 17.3 ms, while brute force is only cheap when the alphabet is small - its inner loop breaks at the first repeat, so it is ~O(n x alphabet) there (2.5 ms at n=3,000 with 36 chars). On all-distinct input it is truly O(n^2): 87.2 ms for n=3,000, while V2 handles n=100,000 in ~11.5 ms.
- **One-line pattern (copy into dsa/patterns.md):** Longest substring without repeats = sliding window + last-seen index map; on a repeat, `left = max(left, last[c] + 1)` - never let `left` move backwards.
